# 04 — Model Debugging

This notebook teaches students how to isolate common failure modes.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

import torch
from model_cnn import SimpleCNN


## Failure mode checklist

- Shape mismatch between layer outputs and linear layer input
- Wrong label range for `CrossEntropyLoss`
- Missing `optimizer.zero_grad()`
- Calling `model.eval()` during training
- Learning rate too large or too small


In [ ]:
model = SimpleCNN(num_classes=10)
x = torch.randn(4, 3, 32, 32)
logits = model(x)
print(logits.shape)


## Sanity checks

A fast debugging routine:
1. Run a single batch forward pass.
2. Run a single batch backward pass.
3. Confirm gradients are non-zero.
4. Overfit on a tiny subset.


In [ ]:
labels = torch.tensor([0, 1, 2, 3])
loss = torch.nn.CrossEntropyLoss()(logits, labels)
loss.backward()

for name, param in model.named_parameters():
    if param.grad is not None:
        print(name, float(param.grad.abs().mean()))
        break


## Student exercise

Open `labs/lab2_fix_broken_model/buggy_model.py` and identify:
- the architectural bug
- the optimization bug
- the data/label bug, if present
